# Pipeline ELT Completo — Sinistros de Trânsito (SP, 2021–2023)

- **Tarefa:** ELT-09 · Notebook ELT completo, rodado do zero
- **Responsável:** Pedro Wellington (`@PedroWellington`) | **Revisor:** Guilherme Calistrato (`@guicalistrato`)
- **Fonte:** 36 arquivos CSV mensais em `data/pessoas_MM-AAAA.csv` (806.977 registros)
- **Destino:** esquema estrela no schema `dw_elt` do PostgreSQL (Supabase)
- **Geração automática:** este notebook foi montado pelo script `scripts/montar_notebook_elt.py` a partir dos arquivos de `sql/elt/`.

No ELT o dado vai cru para o banco e a transformação acontece toda em SQL. Aqui as etapas rodam na ordem, do zero, e cada uma tem o tempo medido.

| Ordem | Etapa | Issue | O que roda | Saída |
| --- | --- | --- | --- | --- |
| 1 | Schemas | ELT-01 | `sql/elt/00_schemas.sql` | os schemas `raw`, `stg` e `dw_elt` |
| 2 | Carga bruta | ELT-02 | COPY dos 36 CSVs | a tabela `raw.pessoas` |
| 3 | Tipos, texto e datas | ELT-03 | `sql/elt/01_stg_tipos_texto_datas.sql` | a tabela `stg.pessoas_tipos_texto_datas` |
| 4 | Nulos e profissão | ELT-04 | `sql/elt/02_stg_nulos_profissao.sql` | a view `stg.pessoas_nulos_profissao` |
| 5 | Dimensões A | ELT-05 | `sql/elt/03_dims_a.sql` | `dim_tempo`, `dim_gravidade`, `dim_profissao` e `dim_local_obito` |
| 6 | Dimensões B | ELT-06 | `sql/elt/04_dims_b.sql` | `dim_perfil_vitima`, `dim_transporte`, `dim_localidade` e `dim_via` |
| 7 | Tabela fato | ELT-07 | `sql/elt/05_fato.sql` | a tabela `dw_elt.fato_pessoa_sinistro` |

A execução inteira leva alguns minutos, a maior parte deles na carga dos CSVs.

## 1. Setup

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)

from src.conexao import obter_engine, testar_conexao

DADOS = RAIZ / "data"
SQL_ELT = RAIZ / "sql" / "elt"

engine = obter_engine()
TEMPOS = []   # (etapa, segundos), para o quadro do final


def rodar_sql(arquivo: str, etapa: str) -> float:
    """Executa um arquivo de sql/elt inteiro, numa transação só, e guarda o tempo."""
    sql = (SQL_ELT / arquivo).read_text(encoding="utf-8")
    engine.dispose()   # conexão nova: tabelas temporárias da etapa anterior não atrapalham
    inicio = time.perf_counter()
    conexao = engine.raw_connection()
    try:
        with conexao.cursor() as cur:
            cur.execute(sql)
        conexao.commit()
    except Exception:
        conexao.rollback()
        raise
    finally:
        conexao.close()
    segundos = time.perf_counter() - inicio
    TEMPOS.append((etapa, segundos))
    print(f"{etapa}: {segundos:.1f} s")
    return segundos


def consulta(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)


_ = testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


## 2. Limpar o que existe

Rodar do zero significa reconstruir as três camadas. Cada arquivo SQL já apaga e recria as próprias
tabelas, mas a fato de uma execução anterior segura as dimensões e impede o `DROP`. Por isso os
schemas são derrubados antes de começar.

Enquanto o notebook não termina, a ELT fica fora do banco. Se parar no meio, é só rodar de novo
desde o começo.

In [2]:
engine.dispose()
conexao = engine.raw_connection()
try:
    with conexao.cursor() as cur:
        for schema in ["dw_elt", "stg", "raw"]:
            cur.execute(f"DROP SCHEMA IF EXISTS {schema} CASCADE")
    conexao.commit()
finally:
    conexao.close()

consulta("""
SELECT nspname AS schema FROM pg_namespace
WHERE nspname IN ('raw', 'stg', 'dw_elt')
ORDER BY nspname
""")

,schema


## 3. Schemas (ELT-01)

Roda `sql/elt/00_schemas.sql`, que cria os schemas `raw`, `stg` e `dw_elt`.

In [3]:
rodar_sql("00_schemas.sql", "Schemas (ELT-01)")

consulta("""
SELECT nspname AS schema FROM pg_namespace
WHERE nspname IN ('raw', 'stg', 'dw_elt')
ORDER BY nspname
""")

Schemas (ELT-01): 1.3 s


,schema
0,dw_elt
1,raw
2,stg


## 4. Carga bruta (ELT-02)

Os 36 CSVs entram crus em `raw.pessoas`: quem lê o arquivo é o próprio PostgreSQL, com `COPY` em
`latin-1` e separador `;`. As 31 colunas ficam como texto e cada linha ganha `ano_origem` e
`arquivo_origem`. Tudo numa transação só, então uma falha no meio não deixa carga pela metade.

In [4]:
TABELA = "raw.pessoas"

arquivos = sorted(DADOS.glob("pessoas_*.csv"))
assert len(arquivos) == 36, f"esperava 36 arquivos, achei {len(arquivos)}"

with open(arquivos[0], encoding="latin-1") as f:
    COLUNAS = f.readline().strip().split(";")

DDL = (
    f"DROP TABLE IF EXISTS {TABELA};\n"
    f"CREATE TABLE {TABELA} (\n"
    + "".join(f"    {c} TEXT,\n" for c in COLUNAS)
    + "    ano_origem SMALLINT NOT NULL,\n"
    + "    arquivo_origem TEXT NOT NULL);"
)
COPY_CSV = "COPY carga FROM STDIN WITH (FORMAT csv, DELIMITER ';', HEADER true, ENCODING 'LATIN1')"

engine.dispose()
inicio = time.perf_counter()
conexao = engine.raw_connection()
try:
    with conexao.cursor() as cur:
        cur.execute(DDL)
        cur.execute(f"CREATE TEMP TABLE carga ({', '.join(f'{c} TEXT' for c in COLUNAS)}) ON COMMIT DROP")
        for arquivo in arquivos:
            cur.execute("TRUNCATE carga")
            with open(arquivo, "rb") as f:
                cur.copy_expert(COPY_CSV, f)
            cur.execute(f"INSERT INTO {TABELA} SELECT carga.*, %s, %s FROM carga",
                        (int(arquivo.stem[-4:]), arquivo.name))
        cur.execute(f"ANALYZE {TABELA}")
    conexao.commit()
except Exception:
    conexao.rollback()
    raise
finally:
    conexao.close()

TEMPOS.append(("Carga bruta (ELT-02)", time.perf_counter() - inicio))
print(f"Carga bruta (ELT-02): {TEMPOS[-1][1]:.1f} s")

consulta(f"SELECT COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos FROM {TABELA}")

Carga bruta (ELT-02): 56.5 s


,linhas,arquivos
0,806977,36


## 5. Tipos, texto e datas (ELT-03)

Roda `sql/elt/01_stg_tipos_texto_datas.sql`, que cria a tabela `stg.pessoas_tipos_texto_datas`.

In [5]:
rodar_sql("01_stg_tipos_texto_datas.sql", "Tipos, texto e datas (ELT-03)")

consulta("""
SELECT COUNT(*) AS linhas FROM stg.pessoas_tipos_texto_datas
""")

Tipos, texto e datas (ELT-03): 157.0 s


,linhas
0,806977


## 6. Nulos e profissão (ELT-04)

Roda `sql/elt/02_stg_nulos_profissao.sql`, que cria a view `stg.pessoas_nulos_profissao`.

In [6]:
rodar_sql("02_stg_nulos_profissao.sql", "Nulos e profissão (ELT-04)")

consulta("""
SELECT COUNT(*) AS linhas FROM stg.pessoas_nulos_profissao
""")

Nulos e profissão (ELT-04): 24.7 s


,linhas
0,806977


## 7. Dimensões A (ELT-05)

Roda `sql/elt/03_dims_a.sql`, que cria `dim_tempo`, `dim_gravidade`, `dim_profissao` e `dim_local_obito`.

In [7]:
rodar_sql("03_dims_a.sql", "Dimensões A (ELT-05)")

consulta("""
SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM dw_elt.dim_tempo
UNION ALL SELECT 'dim_gravidade', COUNT(*) FROM dw_elt.dim_gravidade
UNION ALL SELECT 'dim_profissao', COUNT(*) FROM dw_elt.dim_profissao
UNION ALL SELECT 'dim_local_obito', COUNT(*) FROM dw_elt.dim_local_obito
""")

Dimensões A (ELT-05): 15.7 s


,tabela,linhas
0,dim_tempo,1121
1,dim_gravidade,6
2,dim_profissao,561
3,dim_local_obito,5


## 8. Dimensões B (ELT-06)

Roda `sql/elt/04_dims_b.sql`, que cria `dim_perfil_vitima`, `dim_transporte`, `dim_localidade` e `dim_via`.

In [8]:
rodar_sql("04_dims_b.sql", "Dimensões B (ELT-06)")

consulta("""
SELECT 'dim_perfil_vitima' AS tabela, COUNT(*) AS linhas FROM dw_elt.dim_perfil_vitima
UNION ALL SELECT 'dim_transporte', COUNT(*) FROM dw_elt.dim_transporte
UNION ALL SELECT 'dim_localidade', COUNT(*) FROM dw_elt.dim_localidade
UNION ALL SELECT 'dim_via', COUNT(*) FROM dw_elt.dim_via
""")

Dimensões B (ELT-06): 17.0 s


,tabela,linhas
0,dim_perfil_vitima,1500
1,dim_transporte,25
2,dim_localidade,647
3,dim_via,10


## 9. Tabela fato (ELT-07)

Roda `sql/elt/05_fato.sql`, que cria a tabela `dw_elt.fato_pessoa_sinistro`.

In [9]:
rodar_sql("05_fato.sql", "Tabela fato (ELT-07)")

consulta("""
SELECT COUNT(*) AS linhas, SUM(flag_obito) AS obitos FROM dw_elt.fato_pessoa_sinistro
""")

Tabela fato (ELT-07): 63.7 s


,linhas,obitos
0,806977,15793


## 10. Resultado final

Tempo de cada etapa e as 9 tabelas do modelo estrela no schema `dw_elt`.

In [10]:
tempos = pd.DataFrame(TEMPOS, columns=["etapa", "segundos"]).round({"segundos": 1})
display(tempos)

contagens = consulta("""
SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM dw_elt.dim_tempo
UNION ALL SELECT 'dim_gravidade', COUNT(*) FROM dw_elt.dim_gravidade
UNION ALL SELECT 'dim_perfil_vitima', COUNT(*) FROM dw_elt.dim_perfil_vitima
UNION ALL SELECT 'dim_transporte', COUNT(*) FROM dw_elt.dim_transporte
UNION ALL SELECT 'dim_localidade', COUNT(*) FROM dw_elt.dim_localidade
UNION ALL SELECT 'dim_via', COUNT(*) FROM dw_elt.dim_via
UNION ALL SELECT 'dim_profissao', COUNT(*) FROM dw_elt.dim_profissao
UNION ALL SELECT 'dim_local_obito', COUNT(*) FROM dw_elt.dim_local_obito
UNION ALL SELECT 'fato_pessoa_sinistro', COUNT(*) FROM dw_elt.fato_pessoa_sinistro
""")
display(contagens)

total = float(tempos["segundos"].sum())
linhas_fato = int(contagens.loc[contagens["tabela"] == "fato_pessoa_sinistro", "linhas"].iloc[0])
print(f"ELT completa em {total / 60:.1f} min ({total:.0f} s) | fato com {linhas_fato} linhas")

,etapa,segundos
0,Schemas (ELT-01),1.3
1,Carga bruta (ELT-02),56.5
2,"Tipos, texto e datas (ELT-03)",157.0
3,Nulos e profissão (ELT-04),24.7
4,Dimensões A (ELT-05),15.7
5,Dimensões B (ELT-06),17.0
6,Tabela fato (ELT-07),63.7


,tabela,linhas
0,dim_tempo,1121
1,dim_gravidade,6
2,dim_perfil_vitima,1500
3,dim_transporte,25
4,dim_localidade,647
5,dim_via,10
6,dim_profissao,561
7,dim_local_obito,5
8,fato_pessoa_sinistro,806977


ELT completa em 5.6 min (336 s) | fato com 806977 linhas


## 11. Resumo

A ELT inteira rodou do zero em 5,6 minutos (336 s): schemas, carga dos 36 CSVs em `raw.pessoas`, duas etapas de limpeza em `stg` e a montagem do modelo estrela em `dw_elt`.

| Etapa | Issue | Tempo |
|---|---|---|
| Schemas | ELT-01 | 1,3 s |
| Carga bruta | ELT-02 | 56,5 s |
| Tipos, texto e datas | ELT-03 | 157,0 s |
| Nulos e profissão | ELT-04 | 24,7 s |
| Dimensões A | ELT-05 | 15,7 s |
| Dimensões B | ELT-06 | 17,0 s |
| Tabela fato | ELT-07 | 63,7 s |
| **Total** | | **336 s** |

A etapa mais demorada é a ELT-03, com 157 s, quase metade do tempo todo: é ela que pega as 806.977 linhas de texto cru e converte tipos, datas e acentos de uma vez. Depois vêm a fato (63,7 s) e a carga dos CSVs (56,5 s). As dimensões são rápidas porque mexem com poucas linhas.

As 9 tabelas ficaram com as mesmas contagens da ETL: 806.977 linhas na fato, 15.793 óbitos, e 1.121, 6, 1.500, 25, 647, 10, 561 e 5 nas dimensões.